# ETF Strategy Evaluation & Rapid Prototyping

This notebook allows you to evaluate ETF strategies using production logic from `src/`. It implements performance metrics and portfolio simulation similar to the R `stonksanalysis` workflow.

In [24]:
%load_ext autoreload
%autoreload 2

import sys
import os
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date
import logging
from dotenv import load_dotenv

# Setup environment and paths
load_dotenv("../.env")
sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("..")) # For research/

from research.performance import evaluate_performance, summarize_performance
from research.notebook_utils import set_up_viewer, view
from features import (
    calculate_session_id, 
    add_lags_leads, 
    add_rolling_features, 
    calculate_orders, 
    rally, 
    add_etf_specific_features
)
from get_etf_data import get_etf_data

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

set_up_viewer()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Data Loading
Fetch the full dataset once and cache it in memory.

In [25]:
raw_prices = get_etf_data()
print(f"Loaded {len(raw_prices)} rows from {raw_prices['date'].min()} to {raw_prices['date'].max()}")

INFO:get_etf_data:Fetching ETF list...


INFO:get_etf_data:Found 4880 ETFs/ETVs.


Loaded 2578860 rows from 2023-03-13 to 2026-03-11


## 2. Feature Engineering
Apply the production features to the raw data.

In [26]:
def process_features(df):
    df = calculate_session_id(df)
    df = add_lags_leads(df)
    df = add_rolling_features(df, roll_window=25, short_roll_window=5, corr_window=100)
    df = rally(df)
    
    if "market_cap" not in df.columns:
        df = df.with_columns(pl.lit(0).alias("market_cap"))
        
    df = calculate_orders(df) 
    df = add_etf_specific_features(df)
    return df

prices = process_features(raw_prices)

In [44]:
from itables import show

show(prices[['date','symbol','open','close','sell_rally']].to_pandas())


Loading ITables v2.7.1 from the internet... (need help?)


## 3. Strategy Definition & Backtest
Define your entry/exit signals here.

In [27]:
# --- Example: Rally Strategy ---

def run_rally_backtest(df):
    # 1. Pull execution metrics from T+1 back to T using shift(-1)
    # This aligns signals on Day T with execution on Day T+1
    df = df.with_columns([
        (pl.col("open").shift(-1).over("symbol")).alias("next_open"),
        (pl.col("sell_rally").shift(-1).over("symbol")).alias("next_sell_rally"),
        (pl.col("sell_rally_date").shift(-1).over("symbol")).alias("next_sell_rally_date")
    ])
    
    # 2. Filter for signals on Day T
    signals = df.filter(
        (pl.col("volume") > 1000000) &
        (pl.col("close") > 7) &
        (pl.col("category") == 'equity basket') &
        (pl.col("close") < pl.col("lag1high")) &
        (pl.col("sell_rally_day") > 10) &
        (pl.col("sd_from0") > 0.015)
    )
    
    # 3. Calculate results using Day T+1 targets
    results = signals.select([
        pl.col("date"),
        pl.col("symbol"),
        (pl.col("next_sell_rally") / pl.col("next_open") - 1).alias("outcome"),
        (pl.col("next_sell_rally_date")).alias("sell_date")
    ])
    
    # 4. Apply portfolio constraints
    daily_perf = evaluate_performance(
        results, 
        hold_max=5, 
        buy_per_day_max=1,
        hold_same_max=1 # Prevent duplicate positions in same stock
    )
    
    summary = summarize_performance(daily_perf, results)
    return daily_perf, summary

daily_results, yearly_summary = run_rally_backtest(prices)
display(yearly_summary)


year,avg_return,max_drawdown,total_return,total_trades,days_traded,max_held
i32,f64,f64,f64,u32,u32,i64
2023,-0.0008,-2.173733,3.967555,216,64,5
2024,0.183522,-0.53364,7.40769,377,81,5
2025,0.330339,-1.838972,17.971662,259,72,5
2026,-0.100547,-4.265552,-3.073758,33,15,5


## 4. Visual Analysis
Equity curve and drawdown charts.

In [23]:
def plot_performance(daily_df):
    fig = make_subplots(rows=2, cols=1, 
                        shared_xaxes=True, 
                        vertical_spacing=0.05, 
                        row_heights=[0.7, 0.3])

    # Equity Curve
    fig.add_trace(
        go.Scatter(x=daily_df["date"], y=daily_df["outcome"].cum_sum(), 
                   name="Cumulative Return", line=dict(color="#2ecc71", width=2)),
        row=1, col=1
    )

    # Drawdown
    fig.add_trace(
        go.Scatter(x=daily_df["date"], y=daily_df["drawdown"], 
                   name="Drawdown", fill='tozeroy', line=dict(color="#e74c3c")),
        row=2, col=1
    )

    # Held Positions
    fig.add_trace(
        go.Scatter(x=daily_df["date"], y=daily_df["n_held"], 
                   name="Positions Held", line=dict(color="#3498db", dash='dot')),
        row=1, col=1
    )

    fig.update_layout(
        title="Strategy Performance Overview",
        template="plotly_dark",
        height=800,
        showlegend=True
    )
    
    fig.show()

plot_performance(daily_results)